# 1.01 · MiBici data format normalization

**Phase 1 (data cleaning)** following the [Cookiecutter Data Science](https://cookiecutter-data-science.drivendata.org/using-the-template/) convention: `PHASE.NOTEBOOK-initials-description`.

- **Input:** `data/raw/mibici/` (unmodified CSVs, see notebook `0.01`).
- **Output:** `data/interim/mibici/` as typed Parquet, one file per month, all with the same schema.
- **Scope:** only the *format* (encoding, column names, date formats, dtypes). Value cleaning (duplicates, outliers, nulls) is the next step.

Section 1 shows, with the raw data itself, why this step is needed. The logic lives in `team2_mlops2026.data.raw_data_normalization`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import pyarrow.parquet as pq

from team2_mlops2026.config import MLFLOW_EXPERIMENT_NAME, MLFLOW_TRACKING_URI
from team2_mlops2026.data.raw_data_extraction import RAW_MIBICI_DIR, ingest_tracked
from team2_mlops2026.data.raw_data_normalization import (
    INTERIM_MIBICI_DIR,
    inspect_raw_file,
    normalize_tracked,
    parse_datetimes,
)

pd.set_option("display.max_colwidth", 130)

2026-10-24 15:26:35.115 | INFO     | team2_mlops2026.config:<module>:12 - PROJ_ROOT path is: /Users/xavier/code/tecmty/mlops/fase1Proyecto/MLOps_Team2


## Parameters

The format variants are spread across the whole history (2014 to date), so by default the full history is used (`SINCE = None`, several GB). With a later `SINCE`, some of the examples in section 1 may not appear.

In [3]:
SINCE = None  # YYYY-MM to limit the history; None = everything
WORKERS = 4  # parallel processes for normalization
FORCE = False  # True = re-normalize even if the output is up to date

## 0. Make sure `data/raw` is complete

Idempotent: only downloads what is missing. Tracked in MLflow as `extract-raw-data`, with the raw CSVs as a dataset.

In [4]:
raw_summary = ingest_tracked(since=SINCE)
print(f"MLflow run {raw_summary['mlflow_run_id']} in experiment '{MLFLOW_EXPERIMENT_NAME}' ({MLFLOW_TRACKING_URI})")
print(f"New: {len(raw_summary['new'])} | Already present: {raw_summary['skipped']} | Failed: {len(raw_summary['failed'])}")

/Users/xavier/code/tecmty/mlops/fase1Proyecto/MLOps_Team2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-10-24 15:26:37.179 | INFO     | team2_mlops2026.data.raw_data_extraction:ingest:205 - Found 143 files on the page


Downloading: 0it [00:00, ?it/s]

Downloading: 0it [00:00, ?it/s]

2026-10-24 15:26:44.377 | SUCCESS  | team2_mlops2026.data.raw_data_extraction:ingest:262 - Done: 0 new, 143 skipped, 0 failed


🏃 View run extract-raw-data at: http://127.0.0.1:5001/#/experiments/1/runs/68d2a3f82ec04778b1e70b103b07da74
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1
MLflow run 68d2a3f82ec04778b1e70b103b07da74 in experiment 'mibici' (http://127.0.0.1:5001)
New: 0 | Already present: 143 | Failed: 0


## 1. Why normalization is needed

### 1.1 Format fingerprint of every raw file

For each trips CSV: detected encoding, decoded header, quoting, line ending and the date format of its first row. If MiBici were consistent, this table would have a single row.

In [5]:
trip_files = sorted(RAW_MIBICI_DIR.glob("datos_abiertos_*.csv"))
fingerprints = pd.DataFrame(inspect_raw_file(p) for p in trip_files)

variants = (
    fingerprints.groupby(
        ["encoding", "quoted", "line_ending", "datetime_format", "header"], dropna=False
    )["name"]
    .agg(files="size", first="first", last="last")
    .reset_index()
    .sort_values("first")
)
print(f"{len(trip_files)} raw trip files -> {len(variants)} different formats")
variants

142 raw trip files -> 5 different formats


,encoding,quoted,line_ending,datetime_format,header,files,first,last
3,utf-8,False,LF,%Y-%m-%d %H:%M:%S,"Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id",78,datos_abiertos_2014_12.csv,datos_abiertos_2021_05.csv
2,latin-1,True,LF,%Y-%m-%d %H:%M:%S,"""Viaje_Id"",""Usuario_Id"",""Genero"",""Año_de_nacimiento"",""Inicio_del_viaje"",""Fin_del_viaje"",""Origen_Id"",""Destino_Id""",38,datos_abiertos_2021_06.csv,datos_abiertos_2026_01.csv
4,utf-8,True,LF,%Y-%m-%d %H:%M:%S,"""Viaje_Id"",""Usuario_Id"",""Genero"",""Año_de_nacimiento"",""Inicio_del_viaje"",""Fin_del_viaje"",""Origen_Id"",""Destino_Id""",21,datos_abiertos_2022_12.csv,datos_abiertos_2026_09.csv
0,latin-1,False,CRLF,%d/%m/%Y %H:%M,"Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id",3,datos_abiertos_2023_11.csv,datos_abiertos_2024_10.csv
1,latin-1,True,LF,%Y-%m-%d %H:%M:%S,"""Viaje_Id"",""Usuario_Id"",""Genero"",""A}äe_nacimiento"",""Inicio_del_viaje"",""Fin_del_viaje"",""Origen_Id"",""Destino_Id""",2,datos_abiertos_2024_02.csv,datos_abiertos_2024_05.csv


In [6]:
def example(mask: pd.Series):
    """First raw file matching the mask, or None if the selected history has none."""
    names = fingerprints.loc[mask, "name"]
    if names.empty:
        print("No file with this variant in the selected history (try SINCE = None).")
        return None
    return RAW_MIBICI_DIR / names.iloc[0]

### 1.2 Encoding alternates between UTF-8 and Latin-1

The encoding changes from month to month with no pattern. Reading everything as UTF-8 crashes on Latin-1 files; reading everything as Latin-1 never crashes but **silently** corrupts the UTF-8 ones (`Año` becomes `AÃ±o`). The encoding has to be detected per file.

In [7]:
print("Files per encoding:", fingerprints["encoding"].value_counts().to_dict())

if latin1_file := example(fingerprints["encoding"] == "latin-1"):
    try:
        pd.read_csv(latin1_file, nrows=5)
    except UnicodeDecodeError as e:
        print(f"\nUTF-8 read of {latin1_file.name} fails:\n  {e}")

if utf8_file := example(fingerprints["encoding"] == "utf-8"):
    columns = pd.read_csv(utf8_file, nrows=0, encoding="latin-1").columns
    print(f"\nLatin-1 read of {utf8_file.name} does not fail, but corrupts the header:")
    print(" ", list(columns))

Files per encoding: {'utf-8': 99, 'latin-1': 43}

UTF-8 read of datos_abiertos_2021_06.csv fails:
  'utf-8' codec can't decode byte 0xf1 in position 1: invalid continuation byte

Latin-1 read of datos_abiertos_2014_12.csv does not fail, but corrupts the header:
  ['Viaje_Id', 'Usuario_Id', 'Genero', 'AÃ±o_de_nacimiento', 'Inicio_del_viaje', 'Fin_del_viaje', 'Origen_Id', 'Destino_Id']


### 1.3 Birth year column name corrupted at the source

Even decoded with the right encoding, some files name the column `A}äe_nacimiento`. No encoding recovers it: MiBici published it that way. Since every file has the same 8 columns in the same order, columns are mapped **by position**, while the other 7 names are still validated so a really different file is rejected.

In [8]:
fingerprints.loc[
    ~fingerprints["header"].str.contains("Año_de_nacimiento"), ["name", "encoding", "header"]
]

,name,encoding,header
110,datos_abiertos_2024_02.csv,latin-1,"""Viaje_Id"",""Usuario_Id"",""Genero"",""A}äe_nacimiento"",""Inicio_del_viaje"",""Fin_del_viaje"",""Origen_Id"",""Destino_Id"""
113,datos_abiertos_2024_05.csv,latin-1,"""Viaje_Id"",""Usuario_Id"",""Genero"",""A}äe_nacimiento"",""Inicio_del_viaje"",""Fin_del_viaje"",""Origen_Id"",""Destino_Id"""


### 1.4 Day-first dates silently become wrong dates

A few files use `dd/mm/yyyy H:MM` (no seconds) instead of `yyyy-mm-dd HH:MM:SS`.

In [9]:
fingerprints.loc[
    fingerprints["datetime_format"].str.startswith("%d", na=False),
    ["name", "encoding", "line_ending", "sample_start", "datetime_format"],
]

,name,encoding,line_ending,sample_start,datetime_format
107,datos_abiertos_2023_11.csv,latin-1,CRLF,01/11/2023 0:01,%d/%m/%Y %H:%M
117,datos_abiertos_2024_09.csv,latin-1,CRLF,01/09/2024 0:00,%d/%m/%Y %H:%M
118,datos_abiertos_2024_10.csv,latin-1,CRLF,01/10/2024 0:00,%d/%m/%Y %H:%M


Every trip in a monthly file starts in that month, which gives a direct check. A naive `pd.to_datetime` reads `01/11/2023` as **January 11** (month-first) and turns days above 12 into `NaT`. Parsing with the explicit day-first format keeps every trip inside its month.

In [10]:
def month_check(ts: pd.Series, year: int, month: int) -> pd.Series:
    outside = ts.notna() & ((ts.dt.year != year) | (ts.dt.month != month))
    return pd.Series({"NaT": int(ts.isna().sum()), "outside_file_month": int(outside.sum())})


if dmy_file := example(fingerprints["datetime_format"].str.startswith("%d", na=False)):
    year, month = map(int, dmy_file.stem.split("_")[2:4])
    # Position 4 = Inicio_del_viaje; read as text, exactly as in raw
    raw_start = pd.read_csv(dmy_file, encoding="latin-1", dtype=str, usecols=[4]).iloc[:, 0]

    naive = pd.to_datetime(raw_start, errors="coerce")
    explicit, fmt = parse_datetimes(raw_start)

    print(f"{dmy_file.name}: {len(raw_start):,} trips, all should start in {year}-{month:02d}")
    display(
        pd.DataFrame(
            {
                "naive pd.to_datetime": month_check(naive, year, month),
                f"explicit {fmt}": month_check(explicit, year, month),
            }
        )
    )
    sample = raw_start.drop_duplicates().iloc[[0, -1]]
    display(pd.DataFrame({"raw": sample, "naive": naive[sample.index], "explicit": explicit[sample.index]}))

datos_abiertos_2023_11.csv: 384,464 trips, all should start in 2023-11


,naive pd.to_datetime,explicit %d/%m/%Y %H:%M
NaT,233833,0
outside_file_month,140892,0


,raw,naive,explicit
0,01/11/2023 0:01,2023-01-11 00:01:00,2023-11-01 00:01:00
384463,30/11/2023 23:59,NaT,2023-11-30 23:59:00


## 2. Normalize `raw` -> `interim`

Per file: detect the encoding, map columns by position (validating names), parse dates with the single format that fits the whole file, cast ids and birth year to nullable integers, and write typed Parquet. A file that fails is reported and does not stop the rest.

The step is an MLflow run (`normalize-raw-data`). It logs the raw CSVs as input and the interim trips and stations as outputs, with the same digests that the extraction and cleaning runs log, so the runs chain into one lineage. The encodings and date formats of the files it normalizes are logged as metrics.

In [11]:
summary = normalize_tracked(force=FORCE, workers=WORKERS)

print(f"MLflow run {summary['mlflow_run_id']} in experiment '{MLFLOW_EXPERIMENT_NAME}' ({MLFLOW_TRACKING_URI})")

print(f"Raw files: {summary['found']} | Normalized: {len(summary['normalized'])} | Up to date: {summary['skipped']} | Failed: {len(summary['failed'])}")
if summary["failed"]:
    display(pd.DataFrame(summary["failed"]))
if summary["normalized"]:
    display(
        pd.DataFrame(summary["normalized"])
        .groupby(["kind", "encoding", "datetime_format"], dropna=False)
        .agg(files=("name", "size"), rows=("rows", "sum"))
    )

Normalizing: 0it [00:00, ?it/s]

Normalizing: 0it [00:00, ?it/s]

2026-10-24 15:26:59.936 | SUCCESS  | team2_mlops2026.data.raw_data_normalization:normalize:301 - Normalization done: 0 normalized, 143 skipped, 0 failed


🏃 View run normalize-raw-data at: http://127.0.0.1:5001/#/experiments/1/runs/fce032ebfbf4483da82a7a6342453852
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1
MLflow run fce032ebfbf4483da82a7a6342453852 in experiment 'mibici' (http://127.0.0.1:5001)
Raw files: 143 | Normalized: 0 | Up to date: 143 | Failed: 0


## 3. Validate the `interim` output

Checks run on the Parquet files themselves (not on this run's report), so they also hold when everything was already up to date.

### 3.1 A single schema for every month

In [12]:
trip_parquets = sorted((INTERIM_MIBICI_DIR / "viajes").rglob("*.parquet"))
schemas = pd.Series({p.name: pq.read_schema(p).remove_metadata().to_string() for p in trip_parquets})

print(f"{len(trip_parquets)} interim trip files -> {schemas.nunique()} distinct schema(s)\n")
print(schemas.iloc[0])

142 interim trip files -> 1 distinct schema(s)

Viaje_Id: int64
Usuario_Id: int64
Genero: large_string
Año_de_nacimiento: int64
Inicio_del_viaje: timestamp[ms]
Fin_del_viaje: timestamp[ms]
Origen_Id: int64
Destino_Id: int64


### 3.2 Every trip starts inside its file's month

This is the check that would have caught the day-first dates. Any remaining rows outside their month are a data issue to review in the cleaning step, not a format one.

In [13]:
def month_bounds(path) -> dict:
    start = pd.read_parquet(path, columns=["Inicio_del_viaje"])["Inicio_del_viaje"]
    year, month = map(int, path.stem.split("_")[1:3])
    return {
        "file": path.name,
        "rows": len(start),
        "min_start": start.min(),
        "max_start": start.max(),
        **month_check(start, year, month).to_dict(),
    }


checks = pd.DataFrame(map(month_bounds, trip_parquets))
print(f"Total trips: {checks['rows'].sum():,}")
print(f"Files with trips starting outside their month: {(checks['outside_file_month'] > 0).sum()}")
print(f"Files with null start dates: {(checks['NaT'] > 0).sum()}")
checks.sort_values(["outside_file_month", "NaT"], ascending=False).head(10)

Total trips: 39,078,984
Files with trips starting outside their month: 0
Files with null start dates: 0


,file,rows,min_start,max_start,NaT,outside_file_month
0,viajes_2014_12.parquet,23967,2014-12-01 00:33:47,2014-12-31 23:38:57,0,0
1,viajes_2015_01.parquet,32144,2015-01-01 00:03:06,2015-01-31 23:57:26,0,0
2,viajes_2015_02.parquet,32787,2015-02-01 00:00:03,2015-02-28 23:56:59,0,0
3,viajes_2015_03.parquet,36096,2015-03-01 00:00:41,2015-03-31 23:58:27,0,0
4,viajes_2015_04.parquet,34863,2015-04-01 00:01:28,2015-04-30 23:59:38,0,0
5,viajes_2015_05.parquet,37340,2015-05-01 00:38:25,2015-05-31 23:34:47,0,0
6,viajes_2015_06.parquet,37647,2015-06-01 05:45:25,2015-06-30 23:57:29,0,0
7,viajes_2015_07.parquet,38190,2015-07-01 00:00:00,2015-07-31 23:53:33,0,0
8,viajes_2015_08.parquet,39833,2015-08-01 00:00:23,2015-08-31 23:56:03,0,0
9,viajes_2015_09.parquet,44000,2015-09-01 01:17:15,2015-09-30 23:58:58,0,0


### 3.3 Stations

In [14]:
station_parquets = sorted((INTERIM_MIBICI_DIR / "estaciones").glob("*.parquet"))
stations = pd.read_parquet(station_parquets[-1])
print(station_parquets[-1].name, stations.shape)
print(stations.dtypes.to_string())
stations.head()

nomenclatura_2026_09.parquet (484, 5)
id              Int64
name              str
latitude      float64
longitude     float64
dpcapacity      Int64


,id,name,latitude,longitude,dpcapacity
0,2,(GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.,20.666378,-103.348820,15
1,3,(GDL-002) C. Colonias / Av. Niños héroes,20.667228,-103.366000,15
2,4,(GDL-003) C. Vidrio / Av. Chapultepec,20.667690,-103.368252,19
3,5,(GDL-004) C. Ghilardi /C. Miraflores,20.691847,-103.362549,19
4,6,(GDL-005) C. San Diego /Calzada Independencia,20.681158,-103.339363,11


## Findings and next steps

- Format variants found in raw: mixed UTF-8 / Latin-1 encoding, a corrupted birth year column name, and day-first dates without seconds. All of them are unified in `data/interim/mibici/`.
- _Record here the results of section 3 (schemas, trips outside their month)._
- Next notebook (`1.02`): value cleaning on the interim data (duplicated rows, implausible birth years, null `Genero`, trips with end before start).